In [ ]:
import numpy as np
import torch
from pathlib import Path
import problem_config as cfg
import matplotlib.pyplot as plt

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
dtype = cfg.dtype

# path
base_dir   = Path(cfg.__file__).parent.resolve()
data_dir   = base_dir / "problem_data"
result_dir = base_dir / "result"
result_dir.mkdir(exist_ok=True)

# load data
y_np = np.load(data_dir / "obs_path.npy")          # (T+1, m)
y    = torch.tensor(y_np, dtype=dtype, device=device)

x_true_np = np.load(data_dir / "state_path.npy")          # (T+1, m)

# --- Matrices ---
A = cfg.A.to(device=device, dtype=dtype)
H = cfg.H.to(device=device, dtype=dtype)
Q = cfg.Q.to(device=device, dtype=dtype)
R = cfg.R.to(device=device, dtype=dtype)

n = cfg.DIM_STATE
m = cfg.DIM_OBS
T = y.shape[0]


In [ ]:
import functools
# my lib
from da_utils import KF_update, EnKF_update, KF_with_sample
from da_utils import PF_resample, ResultRecorder, rk4, lorenz96_drift
from iensf import NoiseSchedule, ScoreRep, ReverseSampler, Langevin, post_score, post_score_static


def obs_model(x):
    return x @ H.T

# --- EnSF params ---
ns = NoiseSchedule(ns_type='linear',eps_a=0.001, eps_b=0.001)
dm = ReverseSampler(noise_schedule=ns)
score_rep = ScoreRep(dm=dm, dim_x=cfg.DIM_STATE, obs_model=obs_model, obs_sigma=cfg.OBS_SIGMA)

# --- EnSF params ---
N_ens = 200
Nt = 100
torch.manual_seed(0)


# initial ensemble
m0 = torch.zeros(n, dtype=dtype, device=device)
P0 = torch.eye(n, dtype=dtype, device=device)
L0 = torch.linalg.cholesky(P0)
X  = m0 + torch.randn(N_ens, n, dtype=dtype, device=device) @ L0.T


solver_all = ['SDE_euler', 'ODE_euler', 'DPM_solver']
solver = solver_all[2]

def damp_fn(t):
    return 1-t

def post_score(zt,t, score_prior, score_likelihood):
    return score_prior(zt,t) + score_likelihood(zt,t)


score_ratio = 1
inflation = 1.0
mu_shrink = np.sqrt(1-score_ratio**2)
print(f"score_ratio = {score_ratio}, mu_shrink = {mu_shrink}")
eta1 = 0.5
eta2 = 0.5
N_iter = 3

In [ ]:




LQ = torch.linalg.cholesky(Q)
LR = torch.linalg.cholesky(R)

ms = torch.zeros(T, n, dtype=dtype, device=device)
Ps = torch.zeros(T, n, n, dtype=dtype, device=device)

for k in range(T):
    # save results
    ms[k] = X.mean(dim=0)
    Ps[k] = torch.cov(X.T)

    # ---- Forecast ----
    X = X @ A.T + torch.randn(N_ens, n, dtype=dtype, device=device) @ LQ.T

    # ---- Analysis ----
    X_cov = torch.cov(X.T)
    X_mean = X.mean(dim=0)
    mu_gmm = inflation*mu_shrink * (X - X_mean) + X_mean
    
    
    score_rep.set_obs_value(obs_value=y[k])

    x_T = torch.randn(N_ens, n, dtype=dtype, device=device)
    x_T = (x_T - torch.mean(x_T, dim=0))/torch.std(x_T, dim=0)
    
    score_prior_fn = functools.partial(score_rep.score_gaussian_diffusion,
                                       prior_mean=mu_gmm, prior_cov=X_cov*score_ratio)
    current_sample = X.clone()
    mu_star = torch.mean(current_sample, dim=0)
    cov_star = torch.cov(current_sample.T)

    sample_iter = []
    for i in range(N_iter):
        score_l4 = functools.partial(score_rep.score_likelihood_3_full_generic,
                                    prior_mean=mu_gmm, prior_cov=X_cov*score_ratio,
                                    post_mean=mu_star, post_cov=cov_star)

        post_score_fn = functools.partial(post_score, score_prior=score_prior_fn, score_likelihood=score_l4)
        current_sample = dm.sample_gen(x_T=x_T, score_fn=post_score_fn, Nt=Nt, solver_type=solver)
        
        sample_iter.append(current_sample)
        
        current_mean = torch.mean(current_sample, dim=0)
        current_cov = torch.cov(current_sample.T)
                
        mu_star = (1-eta1)*mu_star + eta1*current_mean
        cov_star = (1-eta2)*cov_star + eta2*current_cov
    
    X = current_sample.clone()
    # # save results
    # ms[k] = X.mean(dim=0)
    # Ps[k] = torch.cov(X.T)


    # print error
    print(f"Step {k+1}/{T} \t error = {np.linalg.norm(ms[k].cpu().numpy() - x_true_np[k]):.4f}")    
    # print(X_cov)
    # print(Ps[k])


# Save results
# out_path = result_dir / "ensf_iter.npz"
# np.savez(out_path,
#          posterior_mean=ms.cpu().numpy(),
#          posterior_cov=Ps.cpu().numpy())
# print(f"Saved EnSF posterior to: {out_path}")


In [ ]:
ms = ms.cpu().numpy()
Ps = Ps.cpu().numpy()

In [ ]:
import matplotlib.pyplot as plt

def kl_gaussian(m0, P0, m1, P1, jitter=1e-9):
    """
    KL( N(m0,P0) || N(m1,P1) ) per step.
    Uses stable solves and adds jitter to keep PD.
    """
    n = P0.shape[-1]
    # jitter for stability
    P0j = P0.copy()
    P1j = P1.copy()
    for M in (P0j, P1j):
        M[...] = 0.5 * (M + M.transpose(0,1))  # symmetrize (if batched later)
    P0j = P0j + jitter * np.eye(n)
    P1j = P1j + jitter * np.eye(n)

    # Cholesky of P1 to get logdet and solve P1^{-1} * X
    L1 = np.linalg.cholesky(P1j)
    logdet_P1 = 2.0 * np.sum(np.log(np.diag(L1)))
    # logdet of P0
    L0 = np.linalg.cholesky(P0j)
    logdet_P0 = 2.0 * np.sum(np.log(np.diag(L0)))

    # trace(P1^{-1} P0) via solves
    # solve L1 * Y = P0  => Y, then solve L1^T * Z = Y => Z = P1^{-1} P0
    Y = np.linalg.solve(L1, P0j)
    Z = np.linalg.solve(L1.T, Y)
    tr_term = np.trace(Z)

    d = (m1 - m0).reshape(-1, 1)  # (n,1)
    # quadratic form d^T P1^{-1} d via solves
    y = np.linalg.solve(L1, d)
    q = y.T @ y

    return float(np.asarray(0.5 * (logdet_P1 - logdet_P0 - n + tr_term + q)).reshape(()))



# load results
base_dir = Path(".").resolve()
result_dir = base_dir / "result"
data_dir = base_dir / "problem_data"
kf = np.load(result_dir / "kf.npz")
ms_kf = kf["posterior_mean"]           # (T, n)
Ps_kf = kf["posterior_cov"]            # (T, n, n)
T, n = ms_kf.shape
t = np.arange(T)



# 
kl_ensf_iter = np.zeros(T)
for k in range(T):
    m0, P0 = ms_kf[k], Ps_kf[k]
    m_temp, P_temp = ms[k], Ps[k]

    kl_ensf_iter[k] = kl_gaussian(m0, P0, m_temp, P_temp)

print(f"Mean KL(KF || EnSF_iter): {kl_ensf_iter.mean():.6f}")

plt.figure(figsize=(7,4))
plt.plot(t, kl_ensf_iter, label="KL(KF || EnSF_iter)")
plt.xlabel("time step")
plt.ylabel("KL divergence")
plt.ylim(0,0.5)
plt.legend()
plt.tight_layout()
plt.show()
